# 178 — Residual Stream Surgery

Surgical modifications to the residual stream: project out directions,
clamp norms, remove component contributions, add steering vectors,
and clamp specific dimensions.

## Why This Matters

Residual stream stream surgery characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_surgery import (
    project_out_direction,
    clamp_residual_norm,
    remove_component_contribution,
    add_steering_at_layer,
    dimension_clamping,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
for layer in range(4):
    result = project_out_direction(model, tokens, direction, layer=layer)
    print(f"Layer {layer}: max_change={result['max_logit_change']:.4f}  "
          f"mean_proj={result['mean_projection']:.4f}")

In [ ]:
for max_norm in [0.1, 0.5, 1.0, 5.0, 100.0]:
    result = clamp_residual_norm(model, tokens, layer=1, max_norm=max_norm)
    print(f"max_norm={max_norm:5.1f}: clamped={result['n_clamped']}/{result['n_total']}  "
          f"max_change={result['max_logit_change']:.4f}")

In [ ]:
print("Removing component contributions:\n")
for layer in range(4):
    for comp in ['attn', 'mlp']:
        result = remove_component_contribution(model, tokens, layer=layer, component=comp)
        print(f"  L{layer} {comp:4s}: norm={result['component_norm']:.4f}  "
              f"KL={result['kl_divergence']:.4f}  max_change={result['max_logit_change']:.4f}")

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(1), (32,))
result = add_steering_at_layer(model, tokens, direction, layer=1, alpha=2.0)
print(f"Steering at layer 1 (alpha=2.0):\n")
print("Top promoted tokens:")
for t in result['top_promoted']:
    print(f"  Token {t['token']:3d}: +{t['logit_change']:.4f}")
print("\nTop demoted tokens:")
for t in result['top_demoted']:
    print(f"  Token {t['token']:3d}: {t['logit_change']:.4f}")

In [ ]:
# Clamp progressively more dimensions
for n_dims in [1, 4, 8, 16, 32]:
    dims = list(range(n_dims))
    result = dimension_clamping(model, tokens, layer=1, dimensions=dims)
    print(f"Clamp {n_dims:2d}/{32} dims: max_change={result['max_logit_change']:.4f}  "
          f"fraction={result['fraction_clamped']:.2f}")